In [1]:
# !pip install ISLP

In [2]:
import pandas as pd
from datetime import datetime

from pandas import DataFrame

In [3]:
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

In [4]:
import statsmodels.api as sm
# from ISLP.utils import load_data      # error
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)

from sklearn.model_selection import train_test_split

from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

## Read training and test datasets

In [5]:
# helper function for reading datatset
def read_data(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d') # convert it to datatime

    return df

In [6]:
df_train = read_data('data/kospi_train.csv')
df_test = read_data('data/kospi_test.csv')

len(df_train), len(df_test)

(986, 244)

In [7]:
# the training dataset has daily KOSPI index from 2019 to 2022
df_train

,Date,Open,Low,High,Close,Volume
0,2019-01-02,2050.550049,2004.270020,2053.449951,2010.000000,326400
1,2019-01-03,2011.810059,1991.650024,2014.719971,1993.699951,428000
2,2019-01-04,1992.400024,1984.530029,2011.560059,2010.250000,409000
3,2019-01-07,2034.239990,2030.900024,2048.060059,2037.099976,440200
4,2019-01-08,2038.680054,2023.589966,2042.699951,2025.270020,397800
...,...,...,...,...,...,...
981,2022-12-23,2325.860107,2311.899902,2333.080078,2313.689941,367000
982,2022-12-26,2312.540039,2304.199951,2321.919922,2317.139893,427600
983,2022-12-27,2327.520020,2321.479980,2335.989990,2332.790039,448300
984,2022-12-28,2296.449951,2276.899902,2296.449951,2280.449951,405700


In [8]:
# the test dataset has daily KOSPI index in 2023
df_test

,Date,Open,Low,High,Close,Volume
0,2023-01-02,2249.949951,2222.370117,2259.879883,2225.669922,346100
1,2023-01-03,2230.979980,2180.669922,2230.979980,2218.679932,410000
2,2023-01-04,2205.979980,2198.820068,2260.060059,2255.979980,412700
3,2023-01-05,2268.199951,2252.969971,2281.389893,2264.649902,430800
4,2023-01-06,2253.399902,2253.270020,2300.620117,2289.969971,398300
...,...,...,...,...,...,...
239,2023-12-21,2598.370117,2587.159912,2610.810059,2600.020020,578300
240,2023-12-22,2617.719971,2599.510010,2621.370117,2599.510010,466000
241,2023-12-26,2609.439941,2594.649902,2612.139893,2602.590088,439500
242,2023-12-27,2599.350098,2590.080078,2613.500000,2613.500000,349700


In [9]:
# a function for residual plot!
# use sns.regplot for fancier plot
def plot_residue(pred, resid):
    """
    inputs: 
        pred - predicted values
        resid - residuals
    """
    
    import seaborn as sns

    res=sns.regplot(x=pred, y=resid, lowess=True, 
            line_kws={'color':'r', 'lw':1},
            scatter_kws={'facecolors':'None', 'edgecolors':'k', 'alpha':0.5})
    XLIM=res.axes.xaxis.get_data_interval()
    #res.axes.hlines(0,XLIM[0], XLIM[1], linestyles='dotted')
    plt.hlines(0,XLIM[0], XLIM[1], linestyles='dotted')
    plt.xlabel('fitted values')
    plt.ylabel('residuals')
    plt.title('Residuals vs. fitted')

## Part 1. Train regression models to predict the next day's `close` using `Open`, `Low`, `High`, `Close`, `Volume` of previous days as predictors using *only* df_train. Cross-validate to select the best model. Evaluate the accuracy of your model using `df_test`.


In [10]:
# for linear regression, preprocessing is needed
# the column means [day 1, day 2, day 3, day 4, day 5]
# the answer is Close of day 6
# to prevent IndexError, the last n-1 columns are trimmed
def get_prev_days_df(df: DataFrame, n_days=3) -> DataFrame: 
    '''make a row using previous n days' variables and the next day's Close
    '''
    df_new = df.copy()

    for j in range(1,n_days):
        for column in ['Open', 'Low', 'High', 'Close', 'Volume']:
            column_name = f'{j}_{column}'
            df_new[column_name] = 0
            for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
    df_new['Answer'] = 0
    for i in range(df_new.shape[0]-n_days): df_new.loc[i, 'Answer'] = df.loc[i+n_days, 'Close']

    df_new = df_new[df_new['Answer'] > 0]

    return df_new

In [11]:
df_train_new = get_prev_days_df(df_train, n_days=5)
df_train_new

C:\Users\hp\AppData\Local\Temp\ipykernel_12736\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2011.81005859375' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12736\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1991.6500244140625' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12736\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2014.719970703125' has dtype incom

,Date,Open,Low,High,Close,Volume,1_Open,1_Low,1_High,1_Close,...,3_Low,3_High,3_Close,3_Volume,4_Open,4_Low,4_High,4_Close,4_Volume,Answer
0,2019-01-02,2050.550049,2004.270020,2053.449951,2010.000000,326400,2011.810059,1991.650024,2014.719971,1993.699951,...,2030.900024,2048.060059,2037.099976,440200,2038.680054,2023.589966,2042.699951,2025.270020,397800,2064.709961
1,2019-01-03,2011.810059,1991.650024,2014.719971,1993.699951,428000,1992.400024,1984.530029,2011.560059,2010.250000,...,2023.589966,2042.699951,2025.270020,397800,2034.189941,2034.189941,2068.229980,2064.709961,386200,2063.280029
2,2019-01-04,1992.400024,1984.530029,2011.560059,2010.250000,409000,2034.239990,2030.900024,2048.060059,2037.099976,...,2034.189941,2068.229980,2064.709961,386200,2065.729980,2057.159912,2072.810059,2063.280029,382900,2075.570068
3,2019-01-07,2034.239990,2030.900024,2048.060059,2037.099976,440200,2038.680054,2023.589966,2042.699951,2025.270020,...,2057.159912,2072.810059,2063.280029,382900,2070.360107,2063.989990,2076.989990,2075.570068,380100,2064.520020
4,2019-01-08,2038.680054,2023.589966,2042.699951,2025.270020,397800,2034.189941,2034.189941,2068.229980,2064.709961,...,2063.989990,2076.989990,2075.570068,380100,2070.489990,2059.459961,2073.939941,2064.520020,432900,2097.179932
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
976,2022-12-16,2329.750000,2326.830078,2360.439941,2360.020020,414200,2350.780029,2342.280029,2358.760010,2352.169922,...,2325.780029,2347.000000,2328.949951,329400,2340.000000,2335.750000,2356.729980,2356.729980,552800,2313.689941
977,2022-12-19,2350.780029,2342.280029,2358.760010,2352.169922,323600,2344.729980,2324.659912,2353.860107,2333.290039,...,2335.750000,2356.729980,2356.729980,552800,2325.860107,2311.899902,2333.080078,2313.689941,367000,2317.139893
978,2022-12-20,2344.729980,2324.659912,2353.860107,2333.290039,358100,2346.389893,2325.780029,2347.000000,2328.949951,...,2311.899902,2333.080078,2313.689941,367000,2312.540039,2304.199951,2321.919922,2317.139893,427600,2332.790039
979,2022-12-21,2346.389893,2325.780029,2347.000000,2328.949951,329400,2340.000000,2335.750000,2356.729980,2356.729980,...,2304.199951,2321.919922,2317.139893,427600,2327.520020,2321.479980,2335.989990,2332.790039,448300,2280.449951


In [12]:
X, Y = df_train_new.drop(columns=['Date','Answer']), df_train_new['Answer']
column_name = list(X.columns)

cv_error = np.zeros(5)

# 1: use all columns
M = sklearn_sm(sm.OLS)
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=df_train_new.shape[0])
cv_error[0] = np.mean(M_CV['test_score'])

# 2: use only previous 3 days
tmp_column = list(filter(lambda x: x.startswith(('2', '3', '4')), column_name))

M = sklearn_sm(sm.OLS)
X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=df_train_new.shape[0])
cv_error[1] = np.mean(M_CV['test_score'])

# 3: use Open, Close, Volume
tmp_column = list(filter(lambda x: x.endswith(('Open', 'Close', 'Volume')), column_name))

M = sklearn_sm(sm.OLS)
X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=df_train_new.shape[0])
cv_error[2] = np.mean(M_CV['test_score'])

# 4: use Open, Close
tmp_column = list(filter(lambda x: x.endswith(('Open', 'Close')), column_name))

M = sklearn_sm(sm.OLS)
X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=df_train_new.shape[0])
cv_error[3] = np.mean(M_CV['test_score'])

# 4: use Open, Close (3 days)
tmp_column = list(filter(lambda x: x.endswith(('Open', 'Close')) and x.startswith(('2','3','4')), column_name))

M = sklearn_sm(sm.OLS)
X = df_train_new[tmp_column]
M_CV = cross_validate(M,
                          X,
                          Y,
                          cv=df_train_new.shape[0])
cv_error[4] = np.mean(M_CV['test_score'])

# 5: use Close, Volume and their interaction
# tmp_column = list(filter(lambda x: x.endswith(('Close', 'Volume')), column_name))
# df_tmp_train = df_train_new[tmp_column].copy()
# df_tmp_train['Close:Volume'] = df_tmp_train['Close'] * df_tmp_train['Volume']

# M = sklearn_sm(sm.OLS)
# X = df_tmp_train
# M_CV = cross_validate(M,
#                           X,
#                           Y,
#                           cv=df_train_new.shape[0])
# cv_error[4] = np.mean(M_CV['test_score'])

cv_error

array([874.08174958, 862.53174085, 859.82535641, 854.71921921,
       851.95652142])

In [13]:
# get fifth method's model

tmp_column = list(filter(lambda x: x.endswith(('Open', 'Close')) and x.startswith(('2','3','4')), column_name))
X = df_train_new[tmp_column]
column_mm = MS(tmp_column)
X_train = column_mm.fit_transform(X)
y_train = Y
model = sm.OLS(y_train, X_train)
results = model.fit()


In [14]:
# get test dataset
df_test_new = get_prev_days_df(df_test, n_days=5)
df_test_new

C:\Users\hp\AppData\Local\Temp\ipykernel_12736\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2230.97998046875' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12736\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2180.669921875' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  for i in range(df_new.shape[0]-j): df_new.loc[i, column_name] = df.loc[i+j, column]
C:\Users\hp\AppData\Local\Temp\ipykernel_12736\3654397654.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2230.97998046875' has dtype incompatib

,Date,Open,Low,High,Close,Volume,1_Open,1_Low,1_High,1_Close,...,3_Low,3_High,3_Close,3_Volume,4_Open,4_Low,4_High,4_Close,4_Volume,Answer
0,2023-01-02,2249.949951,2222.370117,2259.879883,2225.669922,346100,2230.979980,2180.669922,2230.979980,2218.679932,...,2252.969971,2281.389893,2264.649902,430800,2253.399902,2253.270020,2300.620117,2289.969971,398300,2350.189941
1,2023-01-03,2230.979980,2180.669922,2230.979980,2218.679932,410000,2205.979980,2198.820068,2260.060059,2255.979980,...,2253.270020,2300.620117,2289.969971,398300,2315.870117,2312.560059,2351.060059,2350.189941,341100,2351.310059
2,2023-01-04,2205.979980,2198.820068,2260.060059,2255.979980,412700,2268.199951,2252.969971,2281.389893,2264.649902,...,2312.560059,2351.060059,2350.189941,341100,2348.040039,2344.179932,2370.179932,2351.310059,359600,2359.530029
3,2023-01-05,2268.199951,2252.969971,2281.389893,2264.649902,430800,2253.399902,2253.270020,2300.620117,2289.969971,...,2344.179932,2370.179932,2351.310059,359600,2364.050049,2350.360107,2369.659912,2359.530029,368800,2365.100098
4,2023-01-06,2253.399902,2253.270020,2300.620117,2289.969971,398300,2315.870117,2312.560059,2351.060059,2350.189941,...,2350.360107,2369.659912,2359.530029,368800,2376.719971,2358.330078,2377.800049,2365.100098,580100,2386.090088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234,2023-12-14,2547.739990,2532.159912,2549.649902,2544.179932,530100,2558.439941,2555.300049,2574.229980,2563.560059,...,2556.520020,2570.060059,2568.550049,392500,2586.989990,2584.850098,2615.379883,2614.300049,570400,2600.020020
235,2023-12-15,2558.439941,2555.300049,2574.229980,2563.560059,465300,2568.770020,2556.050049,2573.129883,2566.860107,...,2584.850098,2615.379883,2614.300049,570400,2598.370117,2587.159912,2610.810059,2600.020020,578300,2599.510010
236,2023-12-18,2568.770020,2556.050049,2573.129883,2566.860107,383000,2564.810059,2556.520020,2570.060059,2568.550049,...,2587.159912,2610.810059,2600.020020,578300,2617.719971,2599.510010,2621.370117,2599.510010,466000,2602.590088
237,2023-12-19,2564.810059,2556.520020,2570.060059,2568.550049,392500,2586.989990,2584.850098,2615.379883,2614.300049,...,2599.510010,2621.370117,2599.510010,466000,2609.439941,2594.649902,2612.139893,2602.590088,439500,2613.500000


In [15]:
# get test MSE

tmp_column_test = list(filter(lambda x: x.endswith(('Open', 'Close')) and x.startswith(('2','3','4')), column_name))
X = df_test_new[tmp_column_test]
X_test = column_mm.transform(X)
y_test = Y
valid_pred = results.predict(X_test)
np.mean((y_test - valid_pred)**2)

172878.33976577478

## Part 2. Extend the regression model by adding some extra features of your choice. You can use any statistics publicly available. 

In [16]:
# many stock dataset includes increased/decreased price of each day
# as final method is the best, just use Open, Close change in 3 days
def get_dif_df(df_new: DataFrame) -> DataFrame:
    tmp_column = list(filter(lambda x: x.endswith(('Close')), df_new.columns))
    df_dif = df_new[tmp_column].copy()
    df_dif['21_Close_%_dif'] = (df_dif['2_Close'] - df_dif['1_Close']) / df_dif['1_Close'] * 100
    df_dif['32_Close_%_dif'] = (df_dif['3_Close'] - df_dif['2_Close']) / df_dif['2_Close'] * 100
    df_dif['43_Close_%_dif'] = (df_dif['4_Close'] - df_dif['3_Close']) / df_dif['3_Close'] * 100

    tmp_column = list(filter(lambda x: x.endswith('dif') or x.startswith('2_Close'), df_dif.columns))
    df_dif = df_dif[tmp_column].copy()

    return df_dif

In [17]:
df_train_dif = get_dif_df(df_train_new)
df_train_dif

,2_Close,21_Close_%_dif,32_Close_%_dif,43_Close_%_dif
0,2010.250000,0.830117,1.335654,-0.580725
1,2037.099976,1.335654,-0.580725,1.947392
2,2025.270020,-0.580725,1.947392,-0.069256
3,2064.709961,1.947392,-0.069256,0.595655
4,2063.280029,-0.069256,0.595655,-0.532386
...,...,...,...,...
976,2333.290039,-0.802658,-0.186007,1.192813
977,2328.949951,-0.186007,1.192813,-1.826261
978,2356.729980,1.192813,-1.826261,0.149110
979,2313.689941,-1.826261,0.149110,0.675408


In [18]:
# get fifth method's model

X = df_train_dif
column_mm = MS(df_train_dif)
X_train = column_mm.fit_transform(X)
y_train = Y
model = sm.OLS(y_train, X_train)
results_dif = model.fit()

In [19]:
df_test_dif = get_dif_df(df_test_new)
df_test_dif

,2_Close,21_Close_%_dif,32_Close_%_dif,43_Close_%_dif
0,2255.979980,1.681182,0.384308,1.118057
1,2264.649902,0.384308,1.118057,2.629728
2,2289.969971,1.118057,2.629728,0.047661
3,2350.189941,2.629728,0.047661,0.349591
4,2351.310059,0.047661,0.349591,0.236067
...,...,...,...,...
234,2566.860107,0.128729,0.065837,1.781161
235,2568.550049,0.065837,1.781161,-0.546228
236,2614.300049,1.781161,-0.546228,-0.019616
237,2600.020020,-0.546228,-0.019616,0.118487


In [20]:
# get final test MSE

X = df_test_dif
X_test = column_mm.transform(X)
y_test = Y
valid_pred = results_dif.predict(X_test)
np.mean((y_test - valid_pred)**2)

172449.39903082149